# CKKS-VQA: analysis and paper values

This notebook is a **driver**. All model code lives in `src/`; nothing here
redefines it. The notebook

1. checks the environment and runs the fast test suite,
2. optionally reruns any experiment (off by default; the full runs take hours),
3. loads the saved results in `results/`,
4. recomputes every number the manuscript quotes, including the topology
   statistics and bootstrap intervals, and
5. writes `paper/paper_values.tex` (LaTeX macros) and `results/paper_values.csv`.

In [ ]:
# ---- configuration -------------------------------------------------------
RUN_TESTS      = True    # fast pytest tier (~2 s)
RUN_SLOW_TESTS = False   # D4 restart test (~1-2 min)
RUN_SECTOR = False   # full sector study, both preconditioners  (hours)
RUN_DIAG   = False   # precision-plateau diagnostic              (~1.5 h)
RUN_TOPO   = False   # full 2,160-optimisation factorial          (hours)
# The topology statistics below are recomputed from the saved exact CSV, so
# RUN_TOPO is only needed if the training code itself changes.

# Which topology run to analyse, as a folder name under results/.
# Rerunning with a raised evaluation cap:
#   python src/evalmod_factorial_observable_study.py --full --maxfun 1000000 \
#       --output-dir results/factorial_relabelled
TOPO_RUN      = "factorial_relabelled"   # rerun with the mirror cells relabelled by sigma
# Optional earlier run to compare against (None, or a folder name under
# results/ or archive/). Used only for the side-by-side print.
TOPO_PREV_RUN = "factorial_maxfun"       # previous reported run, for the side-by-side print

N_BOOT    = 2000          # bootstrap resamples per contrast
BOOT_SEED = 20260922
ALPHA_FWER = 0.05         # Holm family-wise level;

In [ ]:
import json, os, platform, re, subprocess, sys
from pathlib import Path

import numpy as np
import pandas as pd
import scipy

def find_root(start: Path) -> Path:
    """Locate the project root: a folder containing src/ and results/.

    Order: PROJECT_ROOT below, then the $CKKS_VQA_ROOT environment variable,
    then the working directory and its parents. Set PROJECT_ROOT if Jupyter
    was started outside the project tree.
    """
    PROJECT_ROOT = None      # e.g. r"C:\Users\you\Paper2_CKKS_VQA" or "/home/you/Paper2_CKKS_VQA"

    def ok(p):
        return (p / "src").is_dir() and (p / "results").is_dir()

    tried = []
    for cand in (PROJECT_ROOT, os.environ.get("CKKS_VQA_ROOT")):
        if cand:
            p = Path(cand).expanduser().resolve()
            tried.append(p)
            if ok(p):
                return p
    for p in (start, *start.parents):
        tried.append(p)
        if ok(p):
            return p
    raise FileNotFoundError(chr(10).join([
        "Could not find a folder containing both src/ and results/.",
        f"Working directory: {start}",
        "Looked in: " + ", ".join(str(t) for t in tried[:8]),
        "Set PROJECT_ROOT at the top of this function to your Paper2_CKKS_VQA path.",
    ]))
ROOT = find_root(Path.cwd().resolve())
SRC, RESULTS, PAPER = ROOT / "src", ROOT / "results", ROOT / "paper"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

SECTOR_DIR = RESULTS / "sector_results_full"
DIAG_DIR   = RESULTS / "precision_plateau_diagnostic"
def _run_dir(name):
    for base in (RESULTS, ROOT / "archive"):
        if (base / name).is_dir():
            return base / name
    raise FileNotFoundError(f"no topology run folder named {name!r} under results/ or archive/")

TOPO_DIR      = _run_dir(TOPO_RUN) if not RUN_TOPO else RESULTS / TOPO_RUN
TOPO_PREV_DIR = _run_dir(TOPO_PREV_RUN) if TOPO_PREV_RUN else None

env = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "scipy": scipy.__version__,   # evaluation counts depend on the L-BFGS-B implementation
    "pandas": pd.__version__,
    "platform": platform.platform(),
}
print("project root:", ROOT)
print("topology run :", TOPO_DIR.name + ("" if TOPO_PREV_DIR is None else f"   (compared with {TOPO_PREV_DIR.name})"))
for k, v in env.items():
    print(f"  {k:9s} {v}")
(RESULTS / "environment_analysis.json").write_text(json.dumps(env, indent=2));

## 1. Tests

In [ ]:
if RUN_TESTS:
    proc = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider"], cwd=ROOT,
                          capture_output=True, text=True)
    print(proc.stdout[-1500:] or proc.stderr[-1500:])
    if proc.returncode != 0:
        raise RuntimeError("Fast tests failed; fix the code before trusting any number below.");

if RUN_SLOW_TESTS:
    proc = subprocess.run([sys.executable, "-m", "pytest", "-q", "-m", "slow", "--color=no",
                           "-p", "no:cacheprovider"], cwd=ROOT, capture_output=True, text=True)
    print(proc.stdout[-1500:] or proc.stderr[-1500:])
    if proc.returncode != 0:
        raise RuntimeError("Slow tests failed.")

## 2. Optional reruns (off by default)

In [ ]:
import sector_graded_evalmod as sector
import precision_plateau_diagnostic as diag
import evalmod_factorial_observable_study as topo

if RUN_SECTOR:
    out = sector.main(["--full", "--output-root", str(SECTOR_DIR)])

if RUN_DIAG:
    diag.run_precision_plateau_diagnostic(
        targets=("triangle",), layer_values=(1, 3), kinds=("equivariant",),
        preconditioner_modes=("p0", "matched"), inverse_seeds=40, ftol_seeds=20,
        inverse_maxiter=20_000, ftol_maxiter=50_000, ftols=(1e-12, 1e-15, 1e-18),
        output_dir=str(DIAG_DIR))

if RUN_TOPO:
    # maxfun must match the reported run; the function default is SciPy's 15,000 cap.
    topo.run_factorial_study(output_dir=str(TOPO_DIR), maxfun=1_000_000);

## 3. Value registry

In [ ]:
VALUES, MISSING = [], []

def sci(x, d=3):
    if x == 0:
        return r"\ensuremath{0}"
    m, e = f"{x:.{d}e}".split("e")
    return rf"\ensuremath{{{m}\times10^{{{int(e)}}}}}"

def fixed(x, d=2):
    return rf"\ensuremath{{{x:.{d}f}}}"

def count(x):
    return f"{int(round(x)):,}".replace(",", "{,}")

def pct(x, d=0):
    return rf"\ensuremath{{{100*x:.{d}f}\%}}"

def bits(rmse):
    return -np.log2(rmse)

def put(name, compute, fmt=sci, source=""):
    # Register macro \pv<name>. compute() is called here; failures become ?? .
    assert re.fullmatch(r"[A-Za-z]+", name), f"macro names must be letters only: {name}"
    try:
        v = compute()
        v = float(v) if np.ndim(v) == 0 else v
        if isinstance(v, float) and not np.isfinite(v):
            raise ValueError("non-finite")
        VALUES.append({"macro": name, "value": v, "tex": fmt(v), "source": source})
        return v
    except Exception as exc:
        MISSING.append({"macro": name, "reason": f"{type(exc).__name__}: {exc}"[:160], "source": source})
        VALUES.append({"macro": name, "value": np.nan,
                       "tex": r"\textbf{\textcolor{red}{??}}", "source": source})
        return np.nan

def load(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path.relative_to(ROOT))
    return pd.read_csv(path)

def holm_adjust(p_values):
    """Holm step-down adjustment. Defined here so the notebook does not depend
    on which revision of the factorial module is installed in src/."""
    p = np.asarray(p_values, float)
    m = len(p)
    out, run = np.empty(m), 0.0
    for rank, idx in enumerate(np.argsort(p)):
        run = max(run, (m - rank) * p[idx])
        out[idx] = min(1.0, run)
    return out


## 4. Sector-graded study

Sources: `results/sector_results_full/full_p0` (unweighted decoder, $w_k=1$) and
`full_matched` (decay-matched decoder).

In [ ]:
P0, PM = SECTOR_DIR / "full_p0", SECTOR_DIR / "full_matched"
ex0, exm = load(P0 / "sector_exact_results.csv"), load(PM / "sector_exact_results.csv")
sm0, smm = load(P0 / "sector_exact_summary.csv"), load(PM / "sector_exact_summary.csv")
ranks    = load(P0 / "jacobian_ranks.csv")
checks0, checksm = load(P0 / "prediction_checks.csv"), load(PM / "prediction_checks.csv")
shot0, shotm = load(P0 / "sector_shot_results.csv"), load(PM / "sector_shot_results.csv")
comparison = load(SECTOR_DIR / "full_preconditioning_comparison.csv")
cfg0 = json.loads((P0 / "sector_study_config.json").read_text())

print("seeds:", len(cfg0["seeds"]), " layers:", cfg0["layer_values"],
      " maxiter:", cfg0["maxiter"], " alpha:", cfg0["alpha"], " delta:", cfg0["delta"])
print("\nPrediction checks (p0):")
display(checks0[["prediction", "status", "detail"]])
bad = pd.concat([checks0, checksm]).query("status == 'FAIL' or status.str.startswith('ERROR')", engine="python")
if len(bad):
    print("failing checks:", bad["prediction"].tolist());

In [ ]:
# ---- helpers on the sector frames ------------------------------------------
RANDOM = ["random", "n/a"]

def summ(frame, target, L, method, col="median_nested_rmse"):
    sel = frame[(frame.target == target) & (frame.n_layers == L) &
                (frame.method == method) & frame["init"].isin(RANDOM)]
    if sel.empty:
        raise KeyError(f"no summary row for {target}/L={L}/{method}")
    return float(sel[col].iloc[0])

def classical(frame, target, method, col="nested_rmse"):
    sel = frame[(frame.target == target) & (frame.method == method)]
    if sel.empty:
        raise KeyError(f"no classical row {target}/{method}")
    vals = sel[col].to_numpy(float)
    assert np.ptp(vals) < 1e-12 * max(1.0, abs(vals[0])), "classical rows differ across depths"
    return float(vals[0])

L_MAX = max(cfg0["layer_values"])

# ---- dose response (Table: dense-grid RMSE of LS references) --------------
for tname, tlabel in (("triangle", "Triangle"), ("sawtooth", "Sawtooth"), ("double_sawtooth", "DoubleSawtooth")):
    for m, mlabel in (("zero", "Zero"), ("ols_even", "Even"), ("ols_odd", "Odd"), ("ols_full", "Full")):
        put(f"Dose{tlabel}{mlabel}", lambda t=tname, m=m: classical(ex0, t, m), lambda v: sci(v, 3),
            f"{P0.name}/exact nested_rmse {tname}/{m}")

ratio = put("EvenShareRatio", lambda: classical(ex0, "sawtooth", "ols_odd") / classical(ex0, "sawtooth", "zero"),
            lambda v: fixed(v, 3), "sawtooth ols_odd/zero, nested")
put("EvenSharePct", lambda: ratio ** 2, pct, "square of EvenShareRatio")

# ---- references and plateau (Table: precision diagnostics) ----------------
put("RidgeOddNested",     lambda: classical(ex0, "triangle", "ridge_odd"), source="triangle ridge_odd nested")
put("OLSOddNested",       lambda: classical(ex0, "triangle", "ols_odd"),   source="triangle ols_odd nested")
put("OddReferenceBits",   lambda: bits(classical(ex0, "triangle", "ols_odd")), lambda v: fixed(v, 1),
    "-log2 triangle ols_odd nested")
put("SphericalOddNested", lambda: summ(sm0, "triangle", L_MAX, "spherical_odd"), source="p0 summary")
plateau = put("PlateauRMSE", lambda: summ(sm0, "triangle", L_MAX, "equivariant"),
              source=f"p0 summary equivariant L={L_MAX}")
put("PlateauBits", lambda: bits(plateau), lambda v: fixed(v, 2), "-log2 PlateauRMSE")
put("NullFactor",  lambda: plateau / summ(sm0, "triangle", L_MAX, "spherical_odd"), lambda v: fixed(v, 0),
    "plateau / spherical_odd, default tolerance only")

# ---- evaluation counts ------------------------------------------------------
put("NullEvals", lambda: summ(sm0, "triangle", L_MAX, "spherical_odd", "median_nfev"), count, "p0 summary")
def eq_nfev_range():
    both = pd.concat([sm0, smm])
    sel = both[(both.target == "triangle") & (both.method == "equivariant") & (both["init"] == "random")]
    return sel["median_nfev"].min(), sel["median_nfev"].max()
put("QuantumEvalsMin", lambda: eq_nfev_range()[0], count, "triangle equivariant, both decoders, all L")
put("QuantumEvalsMax", lambda: eq_nfev_range()[1], count, "triangle equivariant, both decoders, all L")

# ---- preconditioning (L = 1, default tolerance) -----------------------------
def comp(col):
    sel = comparison[(comparison.target == "triangle") & (comparison.method == "equivariant") & (comparison.n_layers == 1)]
    return float(sel[col].iloc[0])
put("PrecondGain",       lambda: comp("gap_ratio"), lambda v: fixed(v, 1), "comparison gap_ratio L=1")
put("NfevPzeroLOne",     lambda: comp("nfev_p0"), count, "comparison")
put("NfevMatchedLOne",   lambda: comp("nfev_matched"), count, "comparison")

# ---- optimiser success in the production run ------------------------------
def success_rate(frame):
    v = frame[frame.kind == "variational"]["optimizer_success"].astype(float)
    return v.mean()
put("SectorOptSuccessPzero",   lambda: success_rate(ex0), pct, "p0 exact")
put("SectorOptSuccessMatched", lambda: success_rate(exm), pct, "matched exact")

pd.DataFrame(VALUES).tail(12)

# price of the prior for the triangle, on the training grid, as defined by the
# decomposition of the gap: least-squares optima, odd sector against full space,
# read from the stored results
_odd  = classical(ex0, "triangle", "ols_odd",  col="train_rmse")
_full = classical(ex0, "triangle", "ols_full", col="train_rmse")
print(f"triangle prior price (training grid, LS): odd {_odd:.3e}, full {_full:.3e}, "
      f"price {_odd - _full:.3e}, ratio {_odd / _full:.2f}")
put("PriorTrainOdd",   lambda: _odd, lambda v: sci(v, 3), "triangle LS training RMSE, odd sector")
put("PriorTrainFull",  lambda: _full, lambda v: sci(v, 3), "triangle LS training RMSE, full space")
put("PriorPriceTri",   lambda: _odd - _full, lambda v: sci(v, 2), "price of the prior, triangle, training grid")
put("PriorRatioTri",   lambda: _odd / _full, lambda v: fixed(v, 2), "odd/full LS training RMSE, triangle")


In [ ]:
# ---- lower-bound slacks (P2), recomputed from the exact frame --------------
def min_slack(frame, ceiling, col):
    worst = np.inf
    base = frame[frame["init"] != "warm"]
    for t, g in base.groupby("target"):
        c = g.loc[g.method == ceiling, col]
        if c.empty:
            continue
        vals = g.loc[g.method.isin(["equivariant", "spherical_odd"]), col].dropna()
        if len(vals):
            worst = min(worst, float((vals - c.median()).min()))
    return worst

for label, frame in (("Pzero", ex0), ("Matched", exm)):
    put(f"SlackRMSE{label}",      lambda f=frame: min_slack(f, "ols_odd", "train_rmse"),        source="P2a")
    put(f"SlackObjective{label}", lambda f=frame: min_slack(f, "ridge_odd", "train_objective"), source="P2b")
    _p2 = frame[frame.method.isin(["equivariant", "spherical_odd"]) & (frame["init"] != "warm")]
    _eq = _p2[_p2.method == "equivariant"].shape[0]
    _nu = _p2[_p2.method == "spherical_odd"].drop_duplicates(["target", "seed", "init"]).shape[0]
    put(f"PtwoRows{label}", lambda n=_eq + _nu: n, count,
        "distinct fits checked in P2 (the null is stored once per depth)")

# ---- selection rule (P1) -----------------------------------------------------
put("RawEvenMax", lambda: pd.concat([ex0, exm]).query("method == 'equivariant'")["max_raw_even_expectation"].max(),
    source="P1, worst over both decoders")
for _lab, _f in (("Pzero", ex0), ("Matched", exm)):
    put(f"RawEvenMax{_lab}", lambda f=_f: f.query("method == 'equivariant'")["max_raw_even_expectation"].max(),
        source="P1, this decoder")
    put(f"EquivariantModels{_lab}", lambda f=_f: f.query("method == 'equivariant' and init == 'random'").shape[0],
        count, "trained equivariant models")
put("EquivariantModels", lambda: ex0.query("method == 'equivariant' and init == 'random'").shape[0], count,
    "trained equivariant models, p0");

In [ ]:
# ---- Jacobian ranks (Table: numerically observed rank) --------------------
display(ranks[["n_layers", "kind", "effective_circuit_params", "sector_bound",
               "min_rank", "max_rank", "min_relative_sigma_bound", "rank_rtol"]])

KIND_LABEL = {"generic": "Generic", "sector_mixed": "SectorMixed", "equivariant": "Equivariant"}
L_WORD = {1: "One", 2: "Two", 3: "Three"}
for kind, kl in KIND_LABEL.items():
    for L, lw in L_WORD.items():
        def r(kind=kind, L=L):
            row = ranks[(ranks.kind == kind) & (ranks.n_layers == L)].iloc[0]
            assert row.min_rank == row.max_rank, f"rank varies across sampled points: {kind} L={L}"
            return row.min_rank
        put(f"Rank{kl}L{lw}", r, count, "jacobian_ranks")
    put(f"LiveParams{kl}LOne", lambda kind=kind: ranks[(ranks.kind == kind) & (ranks.n_layers == 1)]
        .effective_circuit_params.iloc[0], count, "effective circuit params L=1")

put("RankRtol", lambda: ranks.rank_rtol.iloc[0], lambda v: sci(v, 0), "rank threshold")
put("SigmaEightMin", lambda: ranks[ranks.kind == "equivariant"].min_relative_sigma_bound.min(),
    source="min sigma_8/sigma_1, equivariant")
put("GenericLOneSigmaMin", lambda: ranks[(ranks.kind == "generic") & (ranks.n_layers == 1)]
    .min_relative_sigma_bound.iloc[0], source="min sigma_16/sigma_1, generic L=1 (fragile cell)")
put("GenericLOneMargin", lambda: ranks[(ranks.kind == "generic") & (ranks.n_layers == 1)]
    .min_relative_sigma_bound.iloc[0] / ranks.rank_rtol.iloc[0], lambda v: fixed(v, 1),
    "margin above threshold");

In [ ]:
# ---- Jacobian ablation: circuit parameters vs decoder scales ---------------
# Separates the quantum degrees of freedom from the classical log-scales,
# using the same estimator and rank criterion as the table above, so the
# `combined` column reproduces it.
ABL_PATH = RESULTS / "jacobian_ablation" / "jacobian_block_ablation.csv"

if ABL_PATH.exists():
    abl = load(ABL_PATH)
else:
    from jacobian_decoder_ablation import jacobian_block_ablation
    abl = jacobian_block_ablation([1, 2, 3], sector.preconditioner(0.0),
                                  n_points=25, rank_rtol=1e-8, seed=11)
    ABL_PATH.parent.mkdir(parents=True, exist_ok=True)
    abl.to_csv(ABL_PATH, index=False)
    print(f"computed and cached -> {ABL_PATH}")

display(abl[["n_layers", "kind", "bound", "rank_circuit_max",
             "rank_decoder_max", "rank_combined_max", "scales_add_dimensions"]])

# The combined column must agree with the rank table above.
for _, _r in abl.iterrows():
    _pub = ranks[(ranks.kind == _r.kind) & (ranks.n_layers == _r.n_layers)].iloc[0]
    assert _r.rank_combined_max == _pub.max_rank, (
        f"ablation disagrees with rank table at {_r.kind} L={_r.n_layers}: "
        f"{_r.rank_combined_max} vs {_pub.max_rank}")
print("combined ranks match the rank table")

_eq3 = abl[(abl.kind == "equivariant") & (abl.n_layers == 3)].iloc[0]
put("RankCircuitEquivLThree", lambda: _eq3.rank_circuit_max, count,
    "circuit-only Jacobian rank, equivariant L=3")
put("RankDecoderEquivLThree", lambda: _eq3.rank_decoder_max, count,
    "decoder-scale-only Jacobian rank, equivariant L=3")
put("ScalesAddDimensions", lambda: int(
        abl[abl.kind == "equivariant"].scales_add_dimensions.max()), count,
    "dimensions the decoder scales add beyond the circuit, equivariant")

if int(abl[abl.kind == "equivariant"].scales_add_dimensions.max()) == 0:
    print("equivariant: decoder scales add no rank")
else:
    print("equivariant: decoder scales add rank")


In [ ]:
# ---- finite-shot accounting (P4), recomputed like-for-like -----------------
def p4(shots):
    s = shots[(shots["init"] == "random") & (shots.predicted_coefficient_noise > 0)]
    keys = ["target", "n_layers", "method", "init", "seed", "total_budget"]
    cells = s.groupby(keys).agg(ms=("coefficient_noise_l2", lambda x: np.mean(np.square(x))),
                                pred=("predicted_coefficient_noise", "first"))
    r = np.sqrt(cells.ms) / cells.pred
    return r.median(), r.quantile(0.1), r.quantile(0.9), len(r)

for label, s in (("Pzero", shot0), ("Matched", shotm)):
    put(f"PfourMedian{label}", lambda s=s: p4(s)[0], lambda v: fixed(v, 3), "RMS/predicted")
    put(f"PfourQTen{label}",   lambda s=s: p4(s)[1], lambda v: fixed(v, 3), "10% quantile")
    put(f"PfourQNinety{label}",lambda s=s: p4(s)[2], lambda v: fixed(v, 3), "90% quantile")
    put(f"PfourCells{label}",  lambda s=s: p4(s)[3], count, "model-budget cells")

def naive(shots):
    s = shots[shots["init"] == "random"]
    out = []
    for _, cell in s.groupby(["target", "n_layers"]):
        med = cell.groupby("method")["coefficient_noise_l2"].median()
        if {"generic", "equivariant"} <= set(med.index) and med["equivariant"] > 0:
            out.append(med["generic"] / med["equivariant"])
    return float(np.median(out))
put("NaiveRatioPzero",   lambda: naive(shot0), lambda v: fixed(v, 2), "generic/equivariant ||dc||")
put("NaiveRatioMatched", lambda: naive(shotm), lambda v: fixed(v, 2), "generic/equivariant ||dc||")

def mse_slope(shots):
    """Median log-log slope of mean MSE degradation vs total budget."""
    s = shots[shots["init"] == "random"]
    out = []
    for _, cell in s.groupby(["target", "n_layers", "method"]):
        g = cell.groupby("total_budget")["nested_mse_degradation"].mean()
        if len(g) < 2 or (g <= 0).any():
            continue
        A = np.vstack([np.log(g.index.to_numpy(float)), np.ones(len(g))]).T
        out.append(np.linalg.lstsq(A, np.log(g.to_numpy()), rcond=None)[0][0])
    return float(np.median(out))

put("PfourSlopePzero",   lambda: mse_slope(shot0), lambda v: fixed(v, 2), "log-log slope of mean MSE degradation")
put("PfourSlopeMatched", lambda: mse_slope(shotm), lambda v: fixed(v, 2), "log-log slope of mean MSE degradation");


## 5. Precision-plateau diagnostic

Direct inversion, conditioning at the best inverse solution, and the `ftol`
sweep. The sweep holds `gtol = 1e-14`, `maxls = 100`, `maxfun = 1e6` fixed
across tolerances; only `ftol` varies.

In [ ]:
inv   = load(DIAG_DIR / "inverse_problem_summary.csv")
cond  = load(DIAG_DIR / "conditioning.csv")
fsum  = load(DIAG_DIR / "ftol_sweep_summary.csv")
fall  = load(DIAG_DIR / "ftol_sweep.csv")
for _df in (fsum, fall):
    _df["ftol"] = _df["ftol"].astype(float)

def at_ftol(df, ftol):
    """Rows of df at exactly this stopping tolerance.
    np.isclose's default atol=1e-8 would treat 1e-12, 1e-15 and 1e-18 as equal,
    so the comparison is relative only."""
    return df[np.isclose(df.ftol, ftol, rtol=1e-6, atol=0.0)]

display(inv[["preconditioner", "n_layers", "best_coefficient_l2", "median_coefficient_l2", "median_nfev"]])
display(fsum[["preconditioner", "n_layers", "ftol", "median_nested_rmse", "median_nested_rmse_gap",
              "median_objective_gap", "median_nfev"]])

PREC = {"p0": "Pzero", "matched": "Matched"}
for p, pl in PREC.items():
    for L in sorted(inv.n_layers.unique()):
        row = lambda p=p, L=L: inv[(inv.preconditioner == p) & (inv.n_layers == L)].iloc[0]
        put(f"InvBest{pl}L{L_WORD[L]}",   lambda row=row: row().best_coefficient_l2,   lambda v: sci(v, 2), "inverse summary")
        put(f"InvMedian{pl}L{L_WORD[L]}", lambda row=row: row().median_coefficient_l2, lambda v: sci(v, 2), "inverse summary")
        c = lambda p=p, L=L: cond[(cond.preconditioner == p) & (cond.n_layers == L)].iloc[0]
        put(f"Cond{pl}L{L_WORD[L]}", lambda c=c: c().condition_number, lambda v: sci(v, 1), "conditioning at best inverse")
        for ftol, fw in ((1e-12, "Default"), (1e-18, "Tight")):
            f = lambda p=p, L=L, ftol=ftol: at_ftol(fsum[(fsum.preconditioner == p) & (fsum.n_layers == L)], ftol).iloc[0]
            put(f"FtolGap{fw}{pl}L{L_WORD[L]}", lambda f=f: f().median_nested_rmse_gap, lambda v: sci(v, 2),
                f"ftol={ftol:g} median nested-RMSE gap")

put("ResidualTangentFraction", lambda: cond[(cond.preconditioner == "p0") &
    (cond.n_layers == cond.n_layers.max())].residual_tangent_fraction.iloc[0], lambda v: fixed(v, 3),
    "residual in tangent space at best p0 inverse")

Lx = int(fsum.n_layers.max())
tight = lambda: at_ftol(fsum[(fsum.preconditioner == "p0") & (fsum.n_layers == Lx)], 1e-18).iloc[0]
tight_rmse = put("TightRMSE", lambda: tight().median_nested_rmse, source=f"p0 L={Lx} ftol=1e-18")
put("TightBits", lambda: bits(tight_rmse), lambda v: fixed(v, 1), "-log2 TightRMSE")
put("TightObjectiveGap", lambda: tight().median_objective_gap, lambda v: sci(v, 2), "median objective gap to ridge")
put("TightNfev", lambda: tight().median_nfev, count, "median nfev at 1e-18")
def frac_ceiling():
    s = at_ftol(fall[(fall.preconditioner == "p0") & (fall.n_layers == Lx)], 1e-18)
    return (s.objective_gap < 1e-12).mean()
put("TightFracCeiling", frac_ceiling, pct, "share of restarts with objective gap < 1e-12 at ftol=1e-18")
put("FtolSeeds", lambda: at_ftol(fall[(fall.preconditioner == "p0") & (fall.n_layers == Lx)], 1e-18)
    .seed.nunique(), count, "restarts per ftol cell")

# Guard against a collapsed tolerance filter: the default and tight gaps must differ in every cell.
for p, pl in PREC.items():
    for L in sorted(inv.n_layers.unique()):
        d = at_ftol(fsum[(fsum.preconditioner == p) & (fsum.n_layers == L)], 1e-12).iloc[0].median_nested_rmse_gap
        t = at_ftol(fsum[(fsum.preconditioner == p) & (fsum.n_layers == L)], 1e-18).iloc[0].median_nested_rmse_gap
        assert d != t, f"ftol filter collapsed for {p} L={L}"
assert len(at_ftol(fall[(fall.preconditioner == "p0") & (fall.n_layers == Lx)], 1e-18)) == \
       fall[(fall.preconditioner == "p0") & (fall.n_layers == Lx)].seed.nunique(), "frac_ceiling pooled over tolerances"

### 5a. Termination diagnostics and the idempotence test

In [ ]:
# ---- termination diagnostics: which stopping condition actually fires -------
# Stopping condition, evaluation counts and projected-gradient norm per
# restart, as recorded by _optimizer_diagnostics in ftol_sweep.csv.
_cell = fall[(fall.preconditioner == "p0") & (fall.n_layers == 3)]

term_summary = _cell.groupby("ftol").apply(lambda g: pd.Series({
    "n":               len(g),
    "median_nfev":     g.nfev.median(),
    "median_nit":      g.nit.median(),
    "median_grad_inf": g.gradient_inf_norm.median(),
    "max_grad_inf":    g.gradient_inf_norm.max(),
    "all_success":     bool(g.success.all()),
    "frac_ceiling":    float((g.objective_gap < 1e-12).mean()),
}), include_groups=False)
display(term_summary)

print("Termination messages, p0 / L=3:")
for _ftol, _g in _cell.groupby("ftol"):
    print(f"  ftol={_ftol:.0e}: {_g.message.value_counts().to_dict()}")

_msg12 = at_ftol(_cell, 1e-12).message
REL_REDUCTION = _msg12.str.contains("RELATIVE REDUCTION", case=False).all()
print(f"\nAll restarts at ftol=1e-12 stop on relative reduction: {REL_REDUCTION}")

def _term(ftol, field):
    return at_ftol(term_summary.reset_index(), ftol)[field].iloc[0]

put("TermNfevDefault",  lambda: _term(1e-12, "median_nfev"),     count, "median nfev, p0 L=3, ftol=1e-12")
put("TermNfevTight",    lambda: _term(1e-18, "median_nfev"),     count, "median nfev, p0 L=3, ftol=1e-18")
put("TermGradDefault",  lambda: _term(1e-12, "median_grad_inf"), lambda v: sci(v, 2), "median |grad|_inf, ftol=1e-12")
put("TermGradTight",    lambda: _term(1e-18, "median_grad_inf"), lambda v: sci(v, 2), "median |grad|_inf, ftol=1e-18")

put("TermFracCeilingDefault", lambda: _term(1e-12, "frac_ceiling"), pct,
    "restarts reaching the ridge optimum at ftol=1e-12")
put("TermFracCeilingMid",     lambda: _term(1e-15, "frac_ceiling"), pct,
    "restarts reaching the ridge optimum at ftol=1e-15")


In [ ]:
# ---- idempotence test: is the study-tolerance solution a fixed point? -----
# Run first:  python src/optimizer_cross_check.py --output-dir results/optimizer_cross_check
# A converged optimisation should be idempotent: restarting the solver from
# its own output, with identical settings, should not improve anything.
IDEM_PATH = RESULTS / "optimizer_cross_check" / "idempotence_test.csv"

if IDEM_PATH.exists():
    idem = load(IDEM_PATH)
    _moved = idem.nested_first - idem.nested_final > 0.1 * idem.nested_first
    _extra = (idem.nfev_total - idem.nfev_first).median()

    display(pd.DataFrame([{
        "restarts":              len(idem),
        "median_nested_first":   idem.nested_first.median(),
        "median_nested_final":   idem.nested_final.median(),
        "median_objective_gain": idem.improvement.median(),
        "frac_improving_10pct":  float(_moved.mean()),
        "median_extra_nfev":     _extra,
    }]))

    put("IdemSeeds",     lambda: len(idem), count,
        "restarts in the idempotence test")
    put("IdemExtraNfev", lambda: _extra, count,
        "median further evaluations before the criterion fires again")
    put("IdemFracMoved", lambda: float(_moved.mean()), pct,
        "restarts improving nested RMSE by more than a tenth")

    # The first leg must reproduce the ftol=1e-12 runs of the sweep.
    _ref = at_ftol(fall[(fall.preconditioner == "p0") & (fall.n_layers == 3)], 1e-12)
    _m = idem.merge(_ref[["seed", "objective"]], on="seed", how="inner")
    _worst = ((_m.f_first - _m.objective).abs() / _m.objective.abs()).max()
    print(f"first leg vs ftol_sweep at ftol=1e-12: worst relative diff {_worst:.2e}")
    assert _worst < 1e-6, ("idempotence first leg does not reproduce the "
                           "published configuration; check OPT_STUDY")
    print("first leg reproduces the ftol=1e-12 sweep")
else:
    print(f"not found: {IDEM_PATH}\n"
          "run:  python src/optimizer_cross_check.py "
          "--output-dir results/optimizer_cross_check")

# ---- criterion probe: restart the endpoint under the tighter rule ---------
# Run:  python src/optimizer_cross_check.py --restart-ftol 1e-18 \
#           --output-dir results/optimizer_cross_check
# A restart discards the L-BFGS curvature history; compare with the
# continuous run at the same tolerance.
TIGHT_RESTART_PATH = RESULTS / "optimizer_cross_check" / "idempotence_test_ftol1e-18.csv"

if TIGHT_RESTART_PATH.exists():
    tr = load(TIGHT_RESTART_PATH)
    _cont = at_ftol(fall[(fall.preconditioner == "p0") & (fall.n_layers == 3)], 1e-18)
    _m = tr.merge(
        _cont[["seed", "nested_rmse"]].rename(columns={"nested_rmse": "nested_continuous"}),
        on="seed")
    _m["ratio"] = _m.nested_final / _m.nested_continuous
    _tr_extra = (tr.nfev_total - tr.nfev_first).median()

    display(pd.DataFrame([{
        "seeds":             len(_m),
        "median_restarted":  _m.nested_final.median(),
        "median_continuous": _m.nested_continuous.median(),
        "median_ratio":      _m.ratio.median(),
        "within_2x":         float((_m.ratio < 2).mean()),
        "median_extra_nfev": _tr_extra,
    }]))

    put("TightRestartRatio", lambda: float(_m.ratio.median()),
        lambda v: f"{v:.2f}",
        "paired ratio, restart at ftol=1e-18 against the continuous run")
    put("TightRestartWithinTwo", lambda: float((_m.ratio < 2).mean()), pct,
        "seeds whose tighter-rule restart lands within 2x of the continuous run")
    put("TightRestartExtraNfev", lambda: _tr_extra, count,
        "median evaluations used by the tighter-rule restart")
    put("TightRestartMoved", lambda: float(
            (tr.nested_first - tr.nested_final > 0.1 * tr.nested_first).mean()),
        pct,
        "restarts at ftol=1e-18 improving nested RMSE by more than a tenth")
    put("TightRestartMedianNested", lambda: float(tr.nested_final.median()),
        lambda v: sci(v, 3),
        "median nested RMSE after restarting at ftol=1e-18")

    print(f"\nfrom the same endpoints: restart at ftol=1e-12 uses "
          f"{_extra:.0f} further evaluations and changes nothing; "
          f"restart at ftol=1e-18 uses {_tr_extra:.0f} and reaches the "
          f"continuous result (paired ratio {_m.ratio.median():.2f}).")
else:
    print(f"not found: {TIGHT_RESTART_PATH}\n"
          "run:  python src/optimizer_cross_check.py --restart-ftol 1e-18 "
          "--output-dir results/optimizer_cross_check")


## 6. Topology factorial

Recomputed from `factorial_exact_results.csv` of the run named in `TOPO_RUN`.
That run uses the corrected factorial script: the mirror circuit and its
decoders are the relabelling sigma of the nearest cells, so (nearest, a) and
(mirror, sigma(a)) are the same optimisation problem (Proposition 1; checked by
`validate_relabelling_equivalence` before any optimisation). The earlier run's
mirror cells were not relabellings, so the side-by-side comparison with
`TOPO_PREV_RUN` is a class-level comparison only. Contrasts are rebuilt here with
their difference vectors (for the bootstrap) and cross-checked against
`topo.paired_factorial_tests`.


In [ ]:
tx = load(TOPO_DIR / "factorial_exact_results.csv")
circ = tx[tx.decoder_type == "factorial_pauli_expectation_4_group_scales"].copy()
cls  = tx[tx.decoder_type == "classical_baseline"].copy()
circ["decoder_matches_topology"] = circ["decoder_matches_topology"].astype(bool)

cfg_path = TOPO_DIR / "factorial_study_config.json"
topo_cfg = json.loads(cfg_path.read_text()) if cfg_path.exists() else {}
print("maxfun:", topo_cfg.get("maxfun", "not recorded (older run: SciPy default 15,000)"),
      " maxiter:", topo_cfg.get("maxiter", "not recorded"))

print("target:", tx.target_type.unique(), " degree:", tx.degree.unique(),
      " K values:", sorted(tx.q0.unique()), " depths:", sorted(tx.n_layers.unique()),
      " seeds:", circ.seed.nunique())

# Recovery and success recomputed locally so an older CSV without these columns still works.
zero = cls[cls.method == "zero_polynomial"].groupby("q0").nested_masked_rmse.first()
ref  = cls[cls.method == "direct_log_monomial"].groupby("q0").nested_masked_rmse.first()
circ["rec"] = (circ.q0.map(zero) - circ.nested_masked_rmse) / (circ.q0.map(zero) - circ.q0.map(ref))
circ["succ"] = circ.nested_masked_rmse <= circ.q0.map(ref) + 1e-4

put("TopoCircuitOpts", lambda: len(circ), count, "circuit optimisations")
distinct = cls.drop_duplicates(["q0", "method", "seed", "nested_masked_rmse"])
put("TopoSphericalFits",  lambda: (distinct.method == "direct_spherical").sum(), count, "distinct fits")
put("TopoLogMonoFits",    lambda: (distinct.method == "direct_log_monomial").sum(), count, "distinct fits")
put("TopoBaselineOpts",   lambda: distinct.method.isin(["direct_spherical", "direct_log_monomial"]).sum(), count,
    "distinct classical optimisations")
put("TopoBaselineRows",   lambda: cls.method.isin(["direct_spherical", "direct_log_monomial"]).sum(), count,
    "classical rows (repeated per depth)")
for K, kw in ((8, "Eight"), (16, "Sixteen")):
    put(f"TopoRefK{kw}",  lambda K=K: ref[K],  lambda v: fixed(v, 4), "direct_log_monomial nested RMSE")
    put(f"TopoZeroK{kw}", lambda K=K: zero[K], lambda v: fixed(v, 4), "zero polynomial nested RMSE")
put("TopoSuccessFraction", lambda: circ.succ.mean(), pct, "within 1e-4 of reference")
put("TopoRMSEMin", lambda: circ.groupby(["q0","n_layers","topology","phase_rule","decoder_alignment"])
    .nested_masked_rmse.median().min(), lambda v: fixed(v, 3), "cell medians")
put("TopoRMSEMax", lambda: circ.groupby(["q0","n_layers","topology","phase_rule","decoder_alignment"])
    .nested_masked_rmse.median().max(), lambda v: fixed(v, 3), "cell medians")
opt = circ.groupby(["q0","n_layers","topology","phase_rule","decoder_alignment"]).optimizer_success.mean()
put("TopoOptSuccessMin", lambda: opt.min(), pct, "per-cell optimizer success")
put("TopoOptSuccessMax", lambda: opt.max(), pct, "per-cell optimizer success")
put("TopoOptSuccessAll", lambda: circ.optimizer_success.mean(), pct, "overall optimizer success")
budget_stop = circ.optimizer_message.fillna("").str.contains("LIMIT", case=False)
put("TopoBudgetStopAll", lambda: budget_stop.mean(), pct, "restarts stopped on an optimiser budget")
put("TopoMaxfun", lambda: float(topo_cfg["maxfun"]) if str(topo_cfg.get("maxfun","")).isdigit() else 15000,
    count, "function-evaluation cap of the analysed run")
# Health check: how often did the optimiser stop on a budget rather than
# converge? Large fractions mean the cells being compared are partly
# budget-limited, which is the confound the rerun is meant to remove.
evals_limit = circ.optimizer_message.fillna("").str.contains("EVALUATIONS EXCEEDS LIMIT", case=False)
iter_limit  = circ.optimizer_message.fillna("").str.contains("ITERATIONS REACHED LIMIT", case=False)
limit = evals_limit | iter_limit
budget = pd.DataFrame({"f_eval_limit": evals_limit.groupby([circ.q0, circ.n_layers]).mean(),
                       "iteration_limit": iter_limit.groupby([circ.q0, circ.n_layers]).mean()}).round(3)
print("\nstopped on a budget, by (K, L):")
display(budget)
if limit.mean() > 0.05:
    print(f"WARNING: {limit.mean():.0%} of restarts stopped on a budget, not on convergence.\n"
          f"         Raise --maxfun (f-eval limit) or maxiter (iteration limit) and rerun,\n"
          f"         or state this explicitly where the affected cells are compared.")
else:
    print(f"OK: only {limit.mean():.1%} of restarts stopped on a budget.")
for K, kw in ((8, "Eight"), (16, "Sixteen")):
    for L in sorted(circ.n_layers.unique()):
        put(f"TopoEvalLimitK{kw}L{L_WORD[L]}", lambda K=K, L=L: limit[(circ.q0 == K) & (circ.n_layers == L)].mean(),
            pct, "share stopped by iteration/evaluation limit");

In [ ]:
# ---- decoder alignment medians (K=8) and the K=16 control regime ----------
m8 = circ[circ.q0 == 8]
put("DecMatchedKEightLOne",    lambda: m8[(m8.n_layers == 1) &  m8.decoder_matches_topology].nested_masked_rmse.median(), lambda v: fixed(v, 4))
put("DecMismatchedKEightLOne", lambda: m8[(m8.n_layers == 1) & ~m8.decoder_matches_topology].nested_masked_rmse.median(), lambda v: fixed(v, 4))
put("DecMatchedKEightDeep",    lambda: m8[(m8.n_layers >= 2) &  m8.decoder_matches_topology].nested_masked_rmse.median(), lambda v: fixed(v, 4))
put("DecMismatchedKEightDeep", lambda: m8[(m8.n_layers >= 2) & ~m8.decoder_matches_topology].nested_masked_rmse.median(), lambda v: fixed(v, 4))
k16 = circ[(circ.q0 == 16) & (circ.n_layers >= 2)].groupby(
    ["n_layers","topology","phase_rule","decoder_alignment"]).nested_masked_rmse.median()
put("TopoKSixteenCentre", lambda: k16.median(), lambda v: fixed(v, 3), "K=16, L>=2 cell medians")
put("TopoKSixteenSpread", lambda: (k16.max() - k16.min()) / k16.median(), lambda v: pct(v, 1), "relative spread")
put("TopoRecoveryKSixteen", lambda: circ[(circ.q0 == 16) & (circ.n_layers >= 2)].rec.median(), lambda v: fixed(v, 2),
    "recovery = (zero-model)/(zero-reference)");

In [ ]:
# ---- paired contrasts with bootstrap intervals ------------------------------
from scipy.stats import wilcoxon
rng_boot = np.random.default_rng(BOOT_SEED)

def boot_ci(d):
    idx = rng_boot.integers(0, len(d), size=(N_BOOT, len(d)))
    med = np.median(d[idx], axis=1)
    return np.quantile(med, [0.025, 0.975])

def test_row(d, ctx):
    d = np.asarray(d, float)
    if np.all(d == 0.0):
        stat, p = 0.0, 1.0
    else:
        t = wilcoxon(d, zero_method="pratt", alternative="two-sided")
        stat, p = float(t.statistic), float(t.pvalue)
    lo, hi = boot_ci(d)
    return {**ctx, "n_pairs": len(d), "n_exact_zero": int(np.sum(d == 0.0)),
            "median_difference": float(np.median(d)), "ci_low": lo, "ci_high": hi,
            "wilcoxon_statistic": stat, "p_value": p}

def contrasts(frame, metric):
    rows = []
    for (K, L, top, dec), g in frame.groupby(["q0", "n_layers", "topology", "decoder_alignment"]):
        pv = g.pivot(index="seed", columns="phase_rule", values=metric).dropna()
        rows.append(test_row(pv.oriented - pv.uniform,
            dict(contrast="oriented_minus_uniform", q0=K, n_layers=L, topology=top, decoder_alignment=dec, metric=metric, phase_rule="both")))
    for (K, L, method), g in frame.groupby(["q0", "n_layers", "method"]):
        top = g.topology.iloc[0]
        pv = g.pivot(index="seed", columns="decoder_alignment", values=metric).dropna()
        others = [c for c in pv.columns if c != top]
        rows.append(test_row(pv[top] - pv[others].mean(axis=1),
            dict(contrast="matched_minus_mean_mismatched", q0=K, n_layers=L, topology=top, decoder_alignment="n/a",
                 metric=metric, phase_rule=g.phase_rule.iloc[0])))
    matched = frame[frame.decoder_matches_topology]
    for (K, L, ph), g in matched.groupby(["q0", "n_layers", "phase_rule"]):
        pv = g.pivot(index="seed", columns="topology", values=metric).dropna()
        for other in ("nearest", "mirror"):
            rows.append(test_row(pv.crossed - pv[other],
                dict(contrast=f"crossed_minus_{other}_matched", q0=K, n_layers=L,
                     topology=f"crossed_vs_{other}", decoder_alignment="matched", metric=metric, phase_rule=ph)))
    return pd.DataFrame(rows)

tests = pd.concat([contrasts(circ, "nested_masked_rmse"), contrasts(circ, "shifted_masked_rmse")], ignore_index=True)
tests["p_holm"] = holm_adjust(tests.p_value.to_numpy())

# Cross-check against the module's own implementation when it provides one.
# Both iterate the same groupbys in the same order, so rows correspond
# one-to-one. (The module's decoder-contrast rows omit the phase rule, so they
# cannot be merged on keys; the rows built here carry it.)
key = ["contrast", "q0", "n_layers", "topology", "decoder_alignment", "metric"]
if hasattr(topo, "paired_factorial_tests"):
    ref_tests = pd.concat([topo.paired_factorial_tests(circ, m)
                           for m in ("nested_masked_rmse", "shifted_masked_rmse")], ignore_index=True)
    assert len(tests) == len(ref_tests), (len(tests), len(ref_tests))
    assert (tests[key].astype(str).to_numpy() == ref_tests[key].astype(str).to_numpy()).all(), "row order differs"
    assert np.allclose(tests.p_value, ref_tests.p_value), "notebook and module p-values disagree"
    print("cross-check against topo.paired_factorial_tests: OK")
else:
    print("src/evalmod_factorial_observable_study.py has no paired_factorial_tests; "
          "the contrasts above are computed here only, with no independent cross-check. "
          "Install the corrected factorial script to enable it.")

tests.to_csv(TOPO_DIR / "factorial_paired_tests_with_bootstrap.csv", index=False)

if TOPO_PREV_DIR is not None:
    prev = load(TOPO_PREV_DIR / "factorial_exact_results.csv")
    prev = prev[prev.decoder_type == "factorial_pauli_expectation_4_group_scales"].copy()
    prev["decoder_matches_topology"] = prev["decoder_matches_topology"].astype(bool)
    prev_tests = pd.concat([contrasts(prev, m) for m in ("nested_masked_rmse", "shifted_masked_rmse")],
                           ignore_index=True)
    prev_tests["p_holm"] = holm_adjust(prev_tests.p_value.to_numpy())
    side = pd.DataFrame({
        TOPO_DIR.name: tests.assign(res=tests.p_holm < ALPHA_FWER).groupby("contrast").res.sum(),
        TOPO_PREV_DIR.name: prev_tests.assign(res=prev_tests.p_holm < ALPHA_FWER).groupby("contrast").res.sum(),
    })
    print("\nresolved contrasts, this run vs earlier run:")
    display(side)
print(tests.groupby(["metric", "contrast"]).size().unstack(0))
print("resolved after Holm:", int((tests.p_holm < ALPHA_FWER).sum()), "of", len(tests))

per_metric = tests.groupby("metric").size()
put("TopoContrastsPerMetric", lambda: per_metric.iloc[0] if per_metric.nunique() == 1 else np.nan, count)
put("TopoPvalues", lambda: len(tests), count, "jointly Holm-adjusted")
for c, cw in (("oriented_minus_uniform", "Phase"), ("matched_minus_mean_mismatched", "Decoder"),
              ("crossed_minus_nearest_matched", "CrossNearest"), ("crossed_minus_mirror_matched", "CrossMirror")):
    put(f"TopoN{cw}", lambda c=c: (tests[(tests.contrast == c) & (tests.metric == "nested_masked_rmse")]).shape[0], count)
put("TopoResolved", lambda: (tests.p_holm < ALPHA_FWER).sum(), count, "p_holm < 0.05")
put("TopoMinPHolm", lambda: tests.p_holm.min(), lambda v: fixed(v, 3), "smallest adjusted p")
put("TopoMinPRaw",  lambda: tests.p_value.min(), lambda v: sci(v, 1), "smallest unadjusted p")
put("TopoAbsDiffMin", lambda: tests.median_difference.abs().min(), lambda v: sci(v, 1), "median differences")
put("TopoAbsDiffMax", lambda: tests.median_difference.abs().max(), lambda v: sci(v, 1), "median differences")
_ci = (tests.ci_low > 0) | (tests.ci_high < 0)
put("TopoCIExcludeZero", lambda: _ci.sum(), count,
    "pointwise 95% bootstrap CIs excluding 0, whole family (not multiplicity-adjusted)")
put("TopoCIExcludeZeroResolved", lambda: (_ci & (tests.p_holm < ALPHA_FWER)).sum(), count,
    "resolved contrasts whose bootstrap CI excludes 0")
put("TopoCIExcludeZeroUnresolved", lambda: (_ci & (tests.p_holm >= ALPHA_FWER)).sum(), count,
    "UNresolved contrasts whose pointwise CI excludes 0 (multiplicity illustration)")
put("TopoPhaseAbsDiffMax", lambda: tests[tests.contrast == "oriented_minus_uniform"].median_difference.abs().max(),
    lambda v: fixed(v, 4), "largest phase-rule effect, resolved or not")

# resolved counts and effect sizes by contrast class
for c, cw in (("oriented_minus_uniform", "Phase"), ("matched_minus_mean_mismatched", "Decoder"),
              ("crossed_minus_nearest_matched", "CrossNearest"), ("crossed_minus_mirror_matched", "CrossMirror")):
    put(f"TopoResolved{cw}", lambda c=c: ((tests.contrast == c) & (tests.p_holm < ALPHA_FWER)).sum(), count,
        f"resolved contrasts, {c}")
for K, kw in ((8, "KEight"), (16, "KSixteen")):
    put(f"TopoResolved{kw}", lambda K=K: ((tests.q0 == K) & (tests.p_holm < ALPHA_FWER)).sum(), count,
        "resolved contrasts in this carry-range condition")
_res = tests[tests.p_holm < ALPHA_FWER]
_top = _res[_res.contrast.str.startswith("crossed")]
put("TopoResolvedEffectMin", lambda: _res.median_difference.abs().min(), lambda v: fixed(v, 4), "resolved effects")
put("TopoResolvedEffectMax", lambda: _res.median_difference.abs().max(), lambda v: fixed(v, 4), "resolved effects")
put("TopoCrossEffectMin", lambda: _top.median_difference.abs().min(), lambda v: fixed(v, 4), "resolved crossed contrasts")
put("TopoCrossEffectMax", lambda: _top.median_difference.abs().max(), lambda v: fixed(v, 4), "resolved crossed contrasts")
put("TopoCrossSameSign", lambda: float(_top.median_difference.gt(0).mean() in (0.0, 1.0)), lambda v: "yes" if v else "no",
    "all resolved crossed contrasts share a sign");


# ---- relabelling check: implementation check, outside the Holm family ----
# The code must pass the equivalence check; the run must leave the nearest and
# crossed cells unchanged; and every nearest cell must match its sigma-image in
# the mirror block up to seed-to-seed variation (the runs start from the same
# parameter vector, not from relabelled ones, so this tests distributions).
if hasattr(topo, "validate_relabelling_equivalence"):
    topo.validate_relabelling_equivalence(layer_values=tuple(sorted(circ.n_layers.unique())))
    print("relabelling equivalence (Proposition 1, as implemented): OK")
if TOPO_PREV_DIR is not None:
    _prev = load(TOPO_PREV_DIR / "factorial_exact_results.csv")
    _prev = _prev[_prev.decoder_type == "factorial_pauli_expectation_4_group_scales"]
    _key = ["q0", "n_layers", "topology", "phase_rule", "decoder_alignment", "seed"]
    _both = circ.merge(_prev, on=_key, suffixes=("", "_prev"))
    _same = _both[_both.topology != "mirror"]
    _worst = float((_same.nested_masked_rmse / _same.nested_masked_rmse_prev - 1).abs().max())
    print(f"nearest and crossed cells vs the previous run: {len(_same)} rows, "
          f"worst relative difference {_worst:.1e}")
    assert _worst < 1e-9, "the corrected run changed cells it should not have"

SIGMA = {"nearest": "mirror", "mirror": "nearest", "crossed": "crossed"}
_rel = []
for a in ("nearest", "mirror", "crossed"):
    n_cells = circ[(circ.topology == "nearest") & (circ.decoder_alignment == a)]
    m_cells = circ[(circ.topology == "mirror") & (circ.decoder_alignment == SIGMA[a])]
    for (K, L, ph), g in n_cells.groupby(["q0", "n_layers", "phase_rule"]):
        gm = m_cells[(m_cells.q0 == K) & (m_cells.n_layers == L) & (m_cells.phase_rule == ph)]
        pv = (g.set_index("seed").nested_masked_rmse.rename("n").to_frame()
              .join(gm.set_index("seed").nested_masked_rmse.rename("m")).dropna())
        _rel.append(test_row(pv.m - pv.n, dict(
            contrast=f"mirror_{SIGMA[a]}_minus_nearest_{a}", q0=K, n_layers=L, phase_rule=ph,
            metric="nested_masked_rmse")))
relabel_tests = pd.DataFrame(_rel)
relabel_tests["p_holm_within"] = holm_adjust(relabel_tests.p_value.to_numpy())
relabel_tests.to_csv(TOPO_DIR / "relabelling_check_tests.csv", index=False)
print(relabel_tests[["contrast", "q0", "n_layers", "phase_rule", "median_difference", "p_value", "p_holm_within"]])
put("TopoRelabelChecks",   lambda: len(relabel_tests), count, "nearest cells vs their sigma-images in the mirror block (implementation check)")
put("TopoRelabelResolved", lambda: (relabel_tests.p_holm_within < ALPHA_FWER).sum(), count,
    "expected 0: corresponding cells are the same problem")

# crossed-vs-nearest and crossed-vs-mirror must now agree in sign class by class
_cn = tests[tests.contrast == "crossed_minus_nearest_matched"].set_index(["q0","n_layers","phase_rule","metric"])
_cm = tests[tests.contrast == "crossed_minus_mirror_matched"].set_index(["q0","n_layers","phase_rule","metric"])
_both = _cn[["median_difference"]].join(_cm[["median_difference"]], lsuffix="_n", rsuffix="_m")
put("TopoCrossSignAgree", lambda: (np.sign(_both.median_difference_n) == np.sign(_both.median_difference_m)).mean(),
    pct, "crossed-vs-nearest and crossed-vs-mirror median differences with the same sign")


In [ ]:
# ---- Feasibility, quality and reproducibility (Mamedaliev et al. 2026) ----
# Definitions follow their Eqs. 5-11 and 17-18, with a 10x10 grid and their
# thresholds (0.70, 1.20, 0.60). The success variable is adapted: their p_min
# is the probability of sampling the global optimum; here
# p = GAP_TOL / (GAP_TOL + gap), so p >= P_THR = 0.5 exactly when the objective
# gap is below GAP_TOL, the ceiling criterion used above. F therefore equals
# frac_ceiling (checked below). Q and R depend on the shape of this map rather
# than the data, so they are computed but not reported. n_max is the largest
# evaluation count in the sweep and enters Q and R only.
GAP_TOL = 1e-12
P_THR   = 0.5
F0, Q0, R0 = 0.70, 1.20, 0.60          # their published thresholds

def mamedaliev_metrics(g, nmax):
    gap = g.objective_gap.clip(lower=0).to_numpy()
    p = GAP_TOL / (GAP_TOL + gap)
    x = (g.nfev.to_numpy() - 1) / (nmax - 1)                  # normalised calls
    y = (1 - p) / (1 - P_THR)
    succ = p >= P_THR
    F = float(succ.mean())                                    # feasibility
    Q = float(np.mean(np.where(succ, 1 / np.hypot(x, y), 0.0)))  # quality
    H, _, _ = np.histogram2d(x, 1 - p, bins=10, range=[[0, 1], [0, 1]])
    q = H.ravel() / H.sum(); q = q[q > 0]
    R = float(1 + (q * np.log(q)).sum() / np.log(100))        # reproducibility
    return F, Q, R

def wilson(k, n, z=1.959964):
    """95% Wilson score interval for a binomial proportion k/n."""
    ph = k / n
    den = 1 + z**2 / n
    ctr = (ph + z**2 / (2 * n)) / den
    half = z * np.sqrt(ph * (1 - ph) / n + z**2 / (4 * n**2)) / den
    return max(0.0, ctr - half), min(1.0, ctr + half)

nmax = fall.nfev.max()
rows = []
for (pc, L, ftol), g in fall.groupby(["preconditioner", "n_layers", "ftol"]):
    F, Q, R = mamedaliev_metrics(g, nmax)
    n = len(g)
    lo, hi = wilson(int(round(F * n)), n)
    rows.append(dict(preconditioner=pc, n_layers=L, ftol=ftol, n=n,
                     F=F, F_lo=lo, F_hi=hi, Q=Q, R=R,
                     frac_ceiling=float((g.objective_gap < GAP_TOL).mean())))
mm = pd.DataFrame(rows).sort_values(["preconditioner", "n_layers", "ftol"])

# Consistency check: F is frac_ceiling under another name.
assert np.allclose(mm.F, mm.frac_ceiling), \
    "feasibility does not match frac_ceiling: the success criteria disagree"

# First step of their selection rule, with uncertainty. A cell is resolved
# only if F0 lies outside its interval.
mm["passes_F"]     = mm.F >= F0
mm["F0_inside_CI"] = (mm.F_lo <= F0) & (F0 <= mm.F_hi)
mm["F_step"] = np.where(mm.F0_inside_CI, "unresolved",
                        np.where(mm.passes_F, "pass", "fail"))

print("feasibility with 95% Wilson intervals, and the first step of their rule:")
display(mm[["preconditioner", "n_layers", "ftol", "n", "F", "F_lo", "F_hi",
            "F_step"]])
print("\nQ and R, computed for completeness only (not comparable to their "
      "thresholds under this adaptation; see the comment above):")
display(mm[["preconditioner", "n_layers", "ftol", "Q", "R"]])

# ---- macros: depth-three unweighted cell at the three tolerances ----
cell = lambda f: mm[(mm.preconditioner == "p0") & (mm.n_layers == 3)
                    & np.isclose(mm.ftol, f, rtol=1e-6, atol=0.0)].iloc[0]
for tag, f in [("Default", 1e-12), ("Mid", 1e-15), ("Tight", 1e-18)]:
    c = cell(f)
    put(f"MmFeas{tag}",   lambda c=c: float(c.F),    pct, f"Mamedaliev F, p0 L=3, ftol={f:g}")
    put(f"MmFeas{tag}Lo", lambda c=c: float(c.F_lo), pct, f"Wilson lower on F, p0 L=3, ftol={f:g}")
    put(f"MmFeas{tag}Hi", lambda c=c: float(c.F_hi), pct, f"Wilson upper on F, p0 L=3, ftol={f:g}")
    put(f"MmQual{tag}",   lambda c=c: float(c.Q), lambda v: f"{v:.2f}",
        f"Mamedaliev Q (not reported), p0 L=3, ftol={f:g}")
    put(f"MmRepr{tag}",   lambda c=c: float(c.R), lambda v: f"{v:.2f}",
        f"Mamedaliev R (not reported), p0 L=3, ftol={f:g}")

matched = mm[mm.preconditioner == "matched"]
put("MmFeasMatchedMax",   lambda: float(matched.F.max()),    pct,
    "largest F over decay-matched cells")
put("MmFeasMatchedMaxHi", lambda: float(matched.F_hi.max()), pct,
    "largest Wilson upper bound on F over decay-matched cells")
put("MmRestarts", lambda: int(mm.n.iloc[0]), count,
    "restarts per cell in the Mamedaliev comparison")

In [ ]:
# ---- Two-optimiser comparison (Thomos & Andronikos 2026); not reported ----
# COBYLA and SPSA on the equivariant, unweighted, L = 3 triangle cell, with the
# objective, bounds, seeds and starting points of the tolerance sweep and
# BUDGET evaluations each (about the L-BFGS-B median at ftol = 1e-18).
import sys, time
SRC = (ROOT / "src") if "ROOT" in globals() else Path("../src")
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
from scipy.optimize import minimize
from sector_graded_evalmod import (build_bundles, evaluate_model, masked_rmse_linf,
                                   model_bounds, preconditioner, random_params,
                                   regularized_loss, ridge_fit)

THOMOS_PATH = RESULTS / "optimizer_cross_check" / "thomos_diagnostic_v2.csv"
BUDGET, SEEDS = 15_000, range(20)
KIND, L, DECAY_P, DELTA, ALPHA = "equivariant", 3, 0.0, 0.06, 1e-8

bundle  = build_bundles(DELTA)["triangle"]
weights = preconditioner(DECAY_P)
ref     = ridge_fit(bundle.train, ALPHA, sector="odd")
ref_obj = regularized_loss(ref, bundle.train, ALPHA)

def objective(x):
    _, _, _, c, _ = evaluate_model(x, KIND, L, weights)
    return regularized_loss(c, bundle.train, ALPHA)

# Bounds: SciPy's convention uses None for "unbounded"; convert to +-inf for
# clipping, and give COBYLA the same object L-BFGS-B receives.
BOUNDS = model_bounds(L)
lo = np.array([-np.inf if b[0] is None else b[0] for b in BOUNDS], dtype=float)
hi = np.array([ np.inf if b[1] is None else b[1] for b in BOUNDS], dtype=float)
assert not np.isnan(lo).any() and not np.isnan(hi).any(), "NaN in bounds"
_x = random_params(L, 0)
assert np.all((_x >= lo) & (_x <= hi)), "start point outside bounds"
assert np.isfinite(objective(np.clip(_x, lo, hi))), "clipped objective not finite"

def spsa(x0, budget, seed, a_target=0.2, c=0.05, alpha=0.602, gamma=0.101):
    """Standard SPSA (Spall 1998). Two evaluations per iteration; the gain a is
    calibrated so the first step has size about a_target, as in Qiskit. The
    best evaluated point is stored before the iterate moves."""
    rng = np.random.default_rng(10_000 + seed)
    n_iter = budget // 2
    A = 0.1 * n_iter
    g_mag = []
    for _ in range(10):
        d = rng.choice([-1.0, 1.0], size=x0.size)
        g_mag.append(abs(objective(np.clip(x0 + c*d, lo, hi))
                         - objective(np.clip(x0 - c*d, lo, hi))) / (2*c))
    a = a_target * (A + 1)**alpha / max(np.mean(g_mag), 1e-300)
    x = x0.copy()
    f0 = objective(x0)
    fbest, xbest, nfev = f0, x0.copy(), 21
    for k in range(n_iter - 11):
        ak, ck = a / (k + 1 + A)**alpha, c / (k + 1)**gamma
        d = rng.choice([-1.0, 1.0], size=x.size)
        xp, xm = np.clip(x + ck*d, lo, hi), np.clip(x - ck*d, lo, hi)
        fp, fm = objective(xp), objective(xm)
        nfev += 2
        if fp < fbest: fbest, xbest = fp, xp.copy()
        if fm < fbest: fbest, xbest = fm, xm.copy()
        x = np.clip(x - ak * (fp - fm) / (2*ck) * d, lo, hi)
    return xbest, fbest, nfev, f0

def run_one(method, seed):
    x0 = random_params(L, seed)
    f0 = objective(x0)
    t0 = time.perf_counter()
    if method == "COBYLA":
        res = minimize(objective, x0, method="COBYLA", bounds=BOUNDS,
                       options=dict(maxiter=BUDGET, rhobeg=0.5, tol=1e-16))
        x, f, nfev = res.x, float(res.fun), int(res.nfev)
    else:
        x, f, nfev, _ = spsa(x0, BUDGET, seed)
    # sanity: the returned point reproduces its value, and improves on x0
    assert np.isclose(objective(x), f, rtol=1e-10, atol=0), f"{method}: point/value mismatch"
    assert f <= f0, f"{method}: ended above its starting value"
    _, _, _, c, _ = evaluate_model(x, KIND, L, weights)
    nested, _ = masked_rmse_linf(c, bundle.nested)
    return dict(optimizer=method, seed=seed, f_start=f0, objective=f,
                objective_gap=f - ref_obj, nested_rmse=nested, nfev=nfev,
                reached_ceiling=bool(f - ref_obj < 1e-12),
                seconds=time.perf_counter() - t0)

if THOMOS_PATH.exists():
    th = pd.read_csv(THOMOS_PATH)
    print(f"loaded {THOMOS_PATH}")
else:
    rows = []
    for method in ("SPSA", "COBYLA"):  
        for s in SEEDS:
            r = run_one(method, int(s))
            rows.append(r)
            print(f"{method} seed {s}: objective {r['f_start']:.2e} -> {r['objective']:.2e}, "
                  f"nested {r['nested_rmse']:.3e}, nfev {r['nfev']}, {r['seconds']:.0f}s")
    th = pd.DataFrame(rows)
    THOMOS_PATH.parent.mkdir(parents=True, exist_ok=True)
    th.to_csv(THOMOS_PATH, index=False)

# L-BFGS-B at the study tolerance, same seeds, from the sweep
lb = at_ftol(fall[(fall.preconditioner == "p0") & (fall.n_layers == L)], 1e-12)
summary = pd.concat([
    th.groupby("optimizer").agg(median_nested=("nested_rmse", "median"),
                                frac_ceiling=("reached_ceiling", "mean"),
                                median_nfev=("nfev", "median")),
    pd.DataFrame({"median_nested": [lb.nested_rmse.median()],
                  "frac_ceiling": [(lb.objective_gap < 1e-12).mean()],
                  "median_nfev": [lb.nfev.median()]},
                 index=["L-BFGS-B (study tolerance)"]),
])
display(summary)

# ---- decision rule: same plateau within a factor of two ----
m_c = summary.loc["COBYLA", "median_nested"]
m_s = summary.loc["SPSA", "median_nested"]
ratio = max(m_c, m_s) / min(m_c, m_s)
same_plateau = ratio < 2.0
neither_reaches = (summary.loc["COBYLA", "frac_ceiling"] < 0.5
                   and summary.loc["SPSA", "frac_ceiling"] < 0.5)
verdict = "model limit" if (same_plateau and neither_reaches) else "optimisation"
budget_stop = {m: float((th[th.optimizer == m].nfev >= BUDGET - 1).mean())
               for m in ("COBYLA", "SPSA")}

print(f"\nCOBYLA median {m_c:.3e}, SPSA median {m_s:.3e}, ratio {ratio:.2f}")
print(f"runs stopped by the budget: COBYLA {budget_stop['COBYLA']:.0%}, "
      f"SPSA {budget_stop['SPSA']:.0%}")
print(f"decision rule: {verdict}")

put("ThomosBudget",        lambda: BUDGET, count, "evaluation budget per optimiser")
put("ThomosCobylaNested",  lambda: float(m_c), lambda v: sci(v, 3), "COBYLA median nested RMSE")
put("ThomosSpsaNested",    lambda: float(m_s), lambda v: sci(v, 3), "SPSA median nested RMSE")
put("ThomosRatio",         lambda: float(ratio), lambda v: f"{v:.2f}", "ratio of the two medians")
put("ThomosCobylaCeiling", lambda: float(summary.loc["COBYLA", "frac_ceiling"]), pct,
    "COBYLA restarts reaching the sector optimum")
put("ThomosSpsaCeiling",   lambda: float(summary.loc["SPSA", "frac_ceiling"]), pct,
    "SPSA restarts reaching the sector optimum")
put("ThomosCobylaBudgetStop", lambda: budget_stop["COBYLA"], pct,
    "COBYLA runs stopped by the evaluation budget")

In [ ]:
# ---- Classical-surrogate comparison (Schreiber, Eisert & Meyer 2023) ----
# The spherical_odd null is the surrogate. It minimises the ridge objective,
# so it is compared with the ridge optimum's nested RMSE; medians within
# TIE_RTOL are reported as ties.
SCHREIBER_TARGETS = ["triangle", "sawtooth", "double_sawtooth"]
GAP_TOL  = 1e-12
TIE_RTOL = 1e-3

def _rand(frame):
    return frame[frame["init"].isin(RANDOM)]

def _verdict(q_med, s_med):
    if np.isclose(q_med, s_med, rtol=TIE_RTOL, atol=0.0):
        return "tie"
    return "quantum better" if q_med < s_med else "no advantage"

rows = []
for dec, frame in (("unweighted", ex0), ("decay-matched", exm)):
    fr = _rand(frame)
    for t in SCHREIBER_TARGETS:
        ft = fr[fr.target == t]
        ridge_nested = classical(frame, t, "ridge_odd")
        ridge_obj    = classical(frame, t, "ridge_odd", col="train_objective")
        for L in sorted(fr.n_layers.dropna().unique()):
            q = ft[(ft.method == "equivariant")   & (ft.n_layers == L)]
            s = ft[(ft.method == "spherical_odd") & (ft.n_layers == L)]
            if q.empty or s.empty:
                continue
            q_med, s_med = q.nested_rmse.median(), s.nested_rmse.median()
            rows.append(dict(
                decoder=dec, target=t, L=int(L),
                ridge_optimum=ridge_nested,
                surrogate_median=s_med,
                quantum_median=q_med,
                their_verdict=_verdict(q_med, s_med),
                unattributed_gap=q_med / s_med,
                surrogate_vs_ridge=s_med / ridge_nested,
                surrogate_obj_gap=float((s.train_objective - ridge_obj).median()),
                surrogate_at_optimum=float(((s.train_objective - ridge_obj) < GAP_TOL).mean()),
            ))
sch = pd.DataFrame(rows)

with pd.option_context("display.float_format", lambda v: f"{v:.3e}"):
    print("surrogate comparison: verdict and quantum/surrogate ratio")
    display(sch[["decoder", "target", "L", "surrogate_median", "quantum_median",
                 "their_verdict", "unattributed_gap"]])
    print("\nDoes the trained surrogate reach the closed-form ridge optimum?")
    display(sch[["decoder", "target", "L", "ridge_optimum", "surrogate_median",
                 "surrogate_vs_ridge", "surrogate_obj_gap", "surrogate_at_optimum"]])

short = sch[(sch.surrogate_vs_ridge > 1.1) | (sch.surrogate_at_optimum < 0.5)]
print(f"\ncells where the trained surrogate falls short of the closed-form optimum: "
      f"{len(short)} of {len(sch)}")

# the triangle gap at L_MAX must match the values behind NullFactor
tri = sch[(sch.decoder == "unweighted") & (sch.target == "triangle") & (sch.L == L_MAX)].iloc[0]
expected = summ(sm0, "triangle", L_MAX, "equivariant") / summ(sm0, "triangle", L_MAX, "spherical_odd")
assert np.isclose(tri.unattributed_gap, expected, rtol=1e-9), \
    "surrogate comparison selects different rows from the summary"
print(f"triangle quantum/surrogate gap at L={L_MAX}: {tri.unattributed_gap:.1f} "
      f"(NullFactor: {expected:.1f})")

# ---- macros ----
_distinct = sch[(sch.decoder == "unweighted") & (sch.L == sch.L.min())]
_short_d = _distinct[(_distinct.surrogate_vs_ridge > 1.1) | (_distinct.surrogate_at_optimum < 0.5)]
put("SchreiberCells",      lambda: len(_distinct), count, "targets in the surrogate comparison (null is depth- and decoder-independent)")
put("SchreiberShortCells", lambda: len(_short_d), count,
    "cells where the surrogate falls short of the ridge optimum")
for L_, tag in ((1, "LOne"), (2, "LTwo"), (3, "LThree")):
    put(f"SchreiberGap{tag}",
        lambda L_=L_: float(sch[(sch.decoder == "unweighted") & (sch.target == "triangle")
                                & (sch.L == L_)].unattributed_gap.iloc[0]),
        lambda v: fixed(v, 0),
        f"quantum/surrogate median nested RMSE, triangle, unweighted, L={L_}")
# share of individual null restarts reaching the ridge optimum (objective within GAP_TOL), by target
for t, tag in (("triangle", "Tri"), ("sawtooth", "Saw"), ("double_sawtooth", "Dbl")):
    v = float(sch[(sch.decoder == "unweighted") & (sch.target == t)].surrogate_at_optimum.iloc[0])
    put(f"SchreiberAtOpt{tag}", lambda v=v: v, pct, f"null restarts at the ridge optimum, {t}")


In [ ]:
# ---- Noisy-target experiment: apparent advantage, attributed ----
# Training targets carry Gaussian noise; every arm is scored on the clean
# nested grid. Arms per noise realisation (seed) include full_ridge, odd_ridge
# (closed form), spherical_odd (trained null), equivariant (quantum).
# Paired comparisons are per seed (same noisy data for every arm), two-sided
# Wilcoxon signed-rank with Pratt zeros, as in the topology study.
from scipy.stats import wilcoxon

REL_EXCESS = 1e-6   # relative excess counted as short of the optimum

NOISY_DIR = RESULTS / "noisy_target"
nt = load(NOISY_DIR / "noisy_target_results.csv").assign(tolerance="study")
if (NOISY_DIR / "noisy_target_results_tight.csv").exists():
    nt = pd.concat([nt, load(NOISY_DIR / "noisy_target_results_tight.csv")
                    .assign(tolerance="tight")], ignore_index=True)
else:
    print("tight-tolerance results not found; the stopping-rule stage is omitted")

def paired(frame, a, b, metric="nested_rmse"):
    w = frame.pivot_table(index="seed", columns="arm", values=metric)
    ratio = (w[a] / w[b]).dropna()
    p = wilcoxon(w[a], w[b], zero_method="pratt").pvalue if len(ratio) else np.nan
    return float(ratio.median()), float(p), int(len(ratio))

rows = []
for (tol, s), g in nt.groupby(["tolerance", "sigma_rel"]):
    rec = dict(tolerance=tol, sigma_rel=s)
    for name, ref in (("naive", "full_ridge"), ("prior", "odd_ridge"), ("null", "spherical_odd")):
        r, p, n = paired(g, "equivariant", ref)
        rec[f"{name}_ratio"], rec[f"{name}_p"] = r, p
    rec["n"] = n
    # early-stopping check: excess of the quantum model's NOISY training objective
    # over the odd-sector optimum, and whether beating the prior on clean data
    # goes with stopping short of it on the training data
    w_obj = g.pivot_table(index="seed", columns="arm", values="objective_noisy")
    w_cln = g.pivot_table(index="seed", columns="arm", values="nested_rmse")
    excess = w_obj.equivariant - w_obj.odd_ridge
    beats = w_cln.equivariant < 0.99 * w_cln.odd_ridge      # beats the prior by more than 1%
    rec["obj_excess_median"] = float(excess.median())
    rec["beats_prior_seeds"] = int(beats.sum())
    # relative criterion: excess of the noisy training objective over the odd-sector
    # optimum, as a fraction of that optimum (an absolute 1e-12 would be effectively
    # zero at high noise, the scale problem diagnosed in the plateau analysis)
    rec["beats_prior_short_of_optimum"] = int((beats & (excess / w_obj.odd_ridge > REL_EXCESS)).sum())
    rec["full_over_odd"] = float((w_cln.full_ridge / w_cln.odd_ridge).median())
    rows.append(rec)
ntsum = pd.DataFrame(rows).sort_values(["tolerance", "sigma_rel"])
with pd.option_context("display.float_format", lambda v: f"{v:.3g}",
                       "display.max_columns", None, "display.width", 200):
    display(ntsum)

# the prediction of component 1: noise-limited error sigma_abs*sqrt(d/n) on n masked points
_b = build_bundles(0.06)["triangle"]
n_train = int(_b.train.target_masked.size)
tri_rms = float(np.sqrt(np.mean(_b.train.target_masked ** 2)))
print(f"\nmasked training points n = {n_train}; target RMS = {tri_rms:.4f}")
for s in (1e-2, 3e-2):
    sa = s * tri_rms
    obs = nt[(nt.tolerance == "study") & np.isclose(nt.sigma_rel, s)] \
            .groupby("arm").nested_rmse.median()
    print(f"sigma_rel={s:g}: predicted full {sa*np.sqrt(16/n_train):.3e} "
          f"(observed {obs.full_ridge:.3e}), odd {sa*np.sqrt(8/n_train):.3e} "
          f"(observed {obs.odd_ridge:.3e})")

# ---- macros ----
def _row(tol, s):
    return ntsum[(ntsum.tolerance == tol) & np.isclose(ntsum.sigma_rel, s)].iloc[0]

pfmt = lambda v: sci(v, 1) if v < 1e-3 else fixed(v, 3)
lo, hi, top = _row("study", 1e-4), _row("study", 1e-2), _row("study", 3e-2)
put("NoisySeeds",        lambda: int(hi.n), count, "noise realisations per level")
put("NoisyTrainPoints",  lambda: n_train, count, "masked training points")
put("NoisyNaiveLow",     lambda: float(lo.naive_ratio), lambda v: fixed(v, 0),
    "quantum/full, sigma_rel=1e-4, study tolerance")
put("NoisyNaiveHigh",    lambda: float(hi.naive_ratio), lambda v: fixed(v, 2),
    "quantum/full, sigma_rel=1e-2, study tolerance")
put("NoisyNaiveHighP",   lambda: float(hi.naive_p), pfmt, "Wilcoxon p, quantum vs full")
put("NoisyNaiveTop",     lambda: float(top.naive_ratio), lambda v: fixed(v, 2),
    "quantum/full, sigma_rel=3e-2, study tolerance")
put("NoisyPriorHigh",    lambda: float(hi.prior_ratio), lambda v: fixed(v, 3),
    "quantum/odd optimum, sigma_rel=1e-2, study tolerance")
put("NoisyPriorHighP",   lambda: float(hi.prior_p), pfmt, "Wilcoxon p, quantum vs odd optimum")
put("NoisyNullHigh",     lambda: float(hi.null_ratio), lambda v: fixed(v, 3),
    "quantum/null, sigma_rel=1e-2, study tolerance")
put("NoisyNullHighP",    lambda: float(hi.null_p), pfmt, "Wilcoxon p, quantum vs null")
put("NoisyFullOverOdd",  lambda: float(hi.full_over_odd), lambda v: fixed(v, 2),
    "full/odd clean RMSE, sigma_rel=1e-2 (sqrt 2 predicted)")
put("NoisyBeatsPrior",   lambda: int(hi.beats_prior_seeds), count,
    "seeds beating the odd optimum on clean data, sigma_rel=1e-2")
put("NoisyBeatsPriorShort", lambda: int(hi.beats_prior_short_of_optimum), count,
    "of those, seeds whose noisy training objective is above the optimum")
if (ntsum.tolerance == "tight").any():
    th = _row("tight", 1e-2)
    put("NoisyTightPrior",  lambda: float(th.prior_ratio), lambda v: fixed(v, 3),
        "quantum/odd optimum, sigma_rel=1e-2, tight tolerance")
    put("NoisyTightPriorP", lambda: float(th.prior_p), pfmt,
        "Wilcoxon p, quantum vs odd optimum, tight tolerance")
mid_s, mid_t = _row("study", 1e-3), _row("tight", 1e-3)
put("NoisyNaiveMidStudy",  lambda: float(mid_s.naive_ratio), lambda v: fixed(v, 2),
    "quantum/full, sigma_rel=1e-3, study tolerance")
put("NoisyNaiveMidStudyP", lambda: float(mid_s.naive_p), pfmt, "Wilcoxon p")
put("NoisyNaiveMidTight",  lambda: float(mid_t.naive_ratio), lambda v: fixed(v, 2),
    "quantum/full, sigma_rel=1e-3, tight tolerance")
put("NoisyNaiveMidTightP", lambda: float(mid_t.naive_p), pfmt, "Wilcoxon p")
_sa = 1e-2 * tri_rms
_obs = nt[(nt.tolerance == "study") & np.isclose(nt.sigma_rel, 1e-2)].groupby("arm").nested_rmse.median()
put("NoisyPredFull", lambda: _sa * np.sqrt(16 / n_train), lambda v: sci(v, 2), "predicted full, 1%")
put("NoisyPredOdd",  lambda: _sa * np.sqrt(8 / n_train),  lambda v: sci(v, 2), "predicted odd, 1%")
put("NoisyObsFull",  lambda: float(_obs.full_ridge), lambda v: sci(v, 2), "observed full, 1%")
put("NoisyObsOdd",   lambda: float(_obs.odd_ridge),  lambda v: sci(v, 2), "observed odd, 1%")
_high = ntsum[ntsum.sigma_rel >= 1e-2]
put("NoisyBeatsPriorTotal", lambda: int(_high.beats_prior_seeds.sum()), count,
    "runs beating the odd optimum by >1% on clean data, sigma_rel >= 1e-2, both tolerances")
put("NoisyBeatsPriorShortTotal", lambda: int(_high.beats_prior_short_of_optimum.sum()), count,
    "of those, runs whose noisy training objective is above the optimum")
put("NoisyNaiveWins", lambda: int((nt[(nt.tolerance == "study") & np.isclose(nt.sigma_rel, 1e-2)]
        .pivot_table(index="seed", columns="arm", values="nested_rmse")
        .pipe(lambda w: w.equivariant < w.full_ridge)).sum()), count,
    "realisations where quantum beats the full fit, sigma_rel=1e-2, study tolerance")

In [ ]:
# ---- Stopping-test scale and the tuned-ridge comparison ----
from scipy.stats import wilcoxon

OS_PATH = RESULTS / "objective_scale" / "objective_scale_results.csv"
osr = load(OS_PATH)
f_ref, fit_term, ridge_term, ref_nested = (float(osr[k].iloc[0])
                                           for k in ("f_ref", "fit_term", "ridge_term", "ref_nested"))
study_ftol = float(osr[osr.arm == "study"].ftol.iloc[0])
w = osr.pivot_table(index="seed", columns="arm", values="nested_rmse")
reached = lambda col: float((w[col] < 1.1 * ref_nested).mean())
os_ratio = (w.normalised / w.study).median()
os_p = wilcoxon(w.normalised, w.study, zero_method="pratt").pvalue
print(f"f_ref {f_ref:.3e}: fit {fit_term:.3e}, ridge {ridge_term:.3e} ({ridge_term/f_ref:.1%})")
print(f"ftol {study_ftol:.0e} = {study_ftol/f_ref:.2e} of f_ref, {study_ftol/fit_term:.2f} of the fit term")
print(f"median nested: study {w.study.median():.3e}, normalised {w.normalised.median():.3e}; "
      f"reached reference: {reached('study'):.0%} -> {reached('normalised'):.0%}; "
      f"paired ratio {os_ratio:.2f}, p = {os_p:.2g}")

put("ObjRef",            lambda: f_ref, lambda v: sci(v, 2), "objective at the odd-sector ridge optimum")
put("ObjFitTerm",        lambda: fit_term, lambda v: sci(v, 2), "data-fit term at the optimum")
put("ObjRidgeShare",     lambda: ridge_term / f_ref, lambda v: pct(v, 1), "ridge share of the objective")
put("FtolOfObjective",   lambda: study_ftol / f_ref, lambda v: sci(v, 1), "study ftol as a fraction of the objective")
put("FtolOfFitTerm",     lambda: study_ftol / fit_term, pct, "study ftol as a fraction of the fit term")
put("NormStudyNested",   lambda: float(w.study.median()), lambda v: sci(v, 3), "median nested, study settings")
put("NormNested",        lambda: float(w.normalised.median()), lambda v: sci(v, 3), "median nested, normalised stopping test")
put("NormReachedStudy",  lambda: reached("study"), pct, "restarts within 10% of the reference, study settings")
put("NormReached",       lambda: reached("normalised"), pct, "restarts within 10% of the reference, normalised")
put("NormP",             lambda: float(os_p), lambda v: sci(v, 1) if v < 1e-3 else fixed(v, 3), "Wilcoxon p")

# tuned full ridge in the noisy-target experiment (study tolerance, 1% noise)
g = nt[(nt.tolerance == "study") & np.isclose(nt.sigma_rel, 1e-2)]
if "full_ridge_cv" in set(g.arm):
    r_cv, p_cv, _ = paired(g, "equivariant", "full_ridge_cv")
    print(f"quantum / tuned full ridge at 1%: {r_cv:.3f} (p = {p_cv:.2g})")
    put("NoisyTunedHigh",  lambda: r_cv, lambda v: fixed(v, 2), "quantum/CV-tuned full ridge, 1% noise")
    put("NoisyTunedHighP", lambda: p_cv, lambda v: sci(v, 1) if v < 1e-3 else fixed(v, 3), "Wilcoxon p")
else:
    print("noisy_target_results.csv has no full_ridge_cv arm; rerun the noisy-target experiment")


# normalised arm against the ftol = 1e-18 sweep, seed by seed
if "sweep_tight_nested" in osr.columns:
    nn = osr[osr.arm == "normalised"].dropna(subset=["sweep_tight_nested"])
    rel = (nn.nested_rmse / nn.sweep_tight_nested - 1.0).abs()
    print(f"normalised vs 1e-18 sweep: {int((rel < 1e-6).sum())} of {len(nn)} seeds identical, worst {rel.max():.1e}")
    put("NormSweepIdentical", lambda: int((rel < 1e-6).sum()), count, "seeds where normalised = 1e-18 sweep")
    put("NormSweepSeeds",     lambda: len(nn), count, "seeds compared")
    put("NormSweepWithinPct", lambda: int((rel < 1e-2).sum()), count, "seeds within 1% of the 1e-18 sweep")
    put("NormSweepWorst",     lambda: float(rel.max()), lambda v: sci(v, 1), "largest relative difference from the sweep")

# data-driven sector selection and the effect of tuning, 1% noise, study tolerance
if "cv_selected" in set(g.arm):
    sel = g[g.arm == "cv_selected"]
    n_odd = int((sel.selected == "odd").sum())
    r_sel, p_sel, _ = paired(g, "equivariant", "cv_selected")
    r_tune, p_tune, _ = paired(g, "full_ridge_cv", "full_ridge")
    print(f"CV picks the odd sector in {n_odd} of {len(sel)}; quantum / selected {r_sel:.3f} (p = {p_sel:.2g}); "
          f"tuned / untuned full {r_tune:.3f} (p = {p_tune:.2g})")
    put("NoisySelectOdd",     lambda: n_odd, count, "realisations where CV selects the odd sector, 1% noise")
    put("NoisySelectedHigh",  lambda: r_sel, lambda v: fixed(v, 3), "quantum/CV-selected fit, 1% noise")
    put("NoisySelectedHighP", lambda: p_sel, lambda v: sci(v, 1) if v < 1e-3 else fixed(v, 3), "Wilcoxon p")
    put("NoisyTuningEffect",  lambda: r_tune, lambda v: fixed(v, 2), "tuned/untuned full ridge, 1% noise")


# sector chosen by cross-validation, at every noise level (study tolerance)
_lv = {0.0: "Zero", 1e-4: "Lo", 1e-3: "Mid", 1e-2: "High", 3e-2: "Top"}
_sel = nt[(nt.tolerance == "study") & (nt.arm == "cv_selected")]
if not _sel.empty:
    for sig, tag in _lv.items():
        rows = _sel[np.isclose(_sel.sigma_rel, sig)]
        if rows.empty:
            continue
        n_odd_s = int((rows.selected == "odd").sum())
        print(f"sigma {sig:g}: CV selects the odd sector in {n_odd_s} of {len(rows)}")
        put(f"NoisySelectOdd{tag}", lambda n=n_odd_s: n, count, f"CV selects odd sector, sigma_rel={sig:g}")

_nm = osr[osr.arm == "normalised"].message.astype(str)
print(f"normalised runs stopped by an iteration or evaluation limit: {_nm.str.contains('LIMIT').mean():.0%}")
put("NormStoppedMaxiter", lambda: float(_nm.str.contains("LIMIT").mean()), pct,
    "normalised runs stopped by an iteration or evaluation limit")

# the same check for the generic family on the sawtooth (full sector), if run
GEN_PATH = RESULTS / "objective_scale" / "objective_scale_generic_sawtooth_L3.csv"
if GEN_PATH.exists():
    gen = load(GEN_PATH)
    g_ref = float(gen.ref_nested.iloc[0])
    gw = gen.pivot_table(index="seed", columns="arm", values="nested_rmse")
    g_reached = lambda col: float((gw[col] < 1.1 * g_ref).mean())
    g_limit = float(gen[gen.arm == "normalised"].message.astype(str).str.contains("LIMIT").mean())
    g_gap = gen.groupby("arm").relative_gap.median()
    print(f"generic/sawtooth: median nested {gw.study.median():.3e} -> {gw.normalised.median():.3e}; "
          f"reached {g_reached('study'):.0%} -> {g_reached('normalised'):.0%}; "
          f"median gap/f_ref {g_gap.study:.1f} -> {g_gap.normalised:.1f}; limit stops {g_limit:.0%}")
    put("GenRefNested",      lambda: g_ref, lambda v: sci(v, 2), "generic/sawtooth full-sector reference, nested")
    put("GenStudyNested",    lambda: float(gw.study.median()), lambda v: sci(v, 2), "generic/sawtooth median, study settings")
    put("GenNormNested",     lambda: float(gw.normalised.median()), lambda v: sci(v, 2), "generic/sawtooth median, normalised")
    put("GenReachedStudy",   lambda: g_reached("study"), pct, "generic/sawtooth within 10% of reference, study")
    put("GenReached",        lambda: g_reached("normalised"), pct, "generic/sawtooth within 10% of reference, normalised")
    put("GenGapStudy",       lambda: float(g_gap.study), lambda v: fixed(v, 1), "median objective gap / f_ref, study")
    put("GenGapNorm",        lambda: float(g_gap.normalised), lambda v: fixed(v, 1), "median objective gap / f_ref, normalised")
    put("GenStoppedLimit",   lambda: g_limit, pct, "generic/sawtooth normalised runs stopped by a limit")
    g_needed = int((gen[gen.arm == "normalised"].nit > cfg0["maxiter"]).sum())
    put("GenNeededIter",     lambda: g_needed, count, "generic/sawtooth normalised runs needing over 3,000 iterations")

    # the matched null for the generic family: spherical_full, same target, same criterion
    _nf = ex0[(ex0.target == "sawtooth") & (ex0.method == "spherical_full")]
    if "init" in _nf.columns:
        _nf = _nf[_nf["init"].isin(RANDOM)]
    _nf = _nf.drop_duplicates("seed")
    if len(_nf):
        nf_share = float((_nf.nested_rmse < 1.1 * g_ref).mean())
        print(f"spherical_full on the sawtooth: {nf_share:.0%} of {len(_nf)} restarts within 10% of the full optimum "
              f"(generic, normalised: {g_reached('normalised'):.0%})")
        put("NullFullSawReached", lambda: nf_share, pct, "spherical_full restarts within 10% of the full optimum, sawtooth")
    else:
        print("no spherical_full rows for the sawtooth in sector_exact_results.csv")


# the generic family on the double sawtooth (full sector), if run
GEN_DBL = RESULTS / "objective_scale" / "objective_scale_generic_double_sawtooth_L3.csv"
if GEN_DBL.exists():
    gd = load(GEN_DBL)
    d_ref = float(gd.ref_nested.iloc[0])
    dw = gd.pivot_table(index="seed", columns="arm", values="nested_rmse")
    d_reached = lambda col: float((dw[col] < 1.1 * d_ref).mean())
    d_gap = gd.groupby("arm").relative_gap.median()
    d_far = int((gd[gd.arm == "normalised"].relative_gap > 1.0).sum())
    print(f"generic/double sawtooth: median nested {dw.study.median():.3e} -> {dw.normalised.median():.3e} "
          f"(optimum {d_ref:.3e}); reached {d_reached('study'):.0%} -> {d_reached('normalised'):.0%}; "
          f"median gap/f_ref {d_gap.study:.2f} -> {d_gap.normalised:.2f}; far from optimum: {d_far}")
    put("GenDblRefNested",    lambda: d_ref, lambda v: sci(v, 2), "generic/double sawtooth full-sector reference, nested")
    put("GenDblStudyNested",  lambda: float(dw.study.median()), lambda v: sci(v, 2), "median, study settings")
    put("GenDblNormNested",   lambda: float(dw.normalised.median()), lambda v: sci(v, 2), "median, normalised")
    put("GenDblReachedStudy", lambda: d_reached("study"), pct, "within 10% of reference, study")
    put("GenDblReached",      lambda: d_reached("normalised"), pct, "within 10% of reference, normalised")
    put("GenDblFar",          lambda: d_far, count, "normalised runs with objective more than twice the optimum")
    _dn = gd[gd.arm == "normalised"]
    put("GenDblStoppedLimit", lambda: float(_dn.message.astype(str).str.contains("LIMIT").mean()), pct,
        "generic/double sawtooth normalised runs stopped by a limit")
    put("GenDblNeededIter",   lambda: int((_dn.nit > cfg0["maxiter"]).sum()), count,
        "generic/double sawtooth normalised runs needing more than the study's iteration cap")


In [ ]:
# ---- Per-family accuracy table (principal study, unweighted decoder) ----
# Median nested RMSE over restarts at the study tolerance, for the three circuit
# families and the classical references, by target and depth. Written to
# paper/family_table.tex as a tabular environment.
FAMILIES = [("generic", r"\textsf{generic}"), ("sector_mixed", r"\textsf{sector\_mixed}"),
            ("equivariant", r"\textsf{equivariant}"), ("spherical_odd", r"null")]
TARGETS = [("triangle", "Triangle"), ("sawtooth", "Sawtooth"), ("double_sawtooth", "Double sawtooth")]

def tex_sci(v, digits=2):
    m, e = f"{v:.{digits - 1}e}".split("e")
    return rf"${m}\times10^{{{int(e)}}}$"

fam = []
for t, tname in TARGETS:
    for L in sorted(cfg0["layer_values"]):
        row = {"target": t, "L": L}
        for m, _ in FAMILIES:
            row[m] = summ(sm0, t, L, m)
        row["ols_odd"], row["ols_full"] = classical(ex0, t, "ols_odd"), classical(ex0, t, "ols_full")
        fam.append(row)
fam = pd.DataFrame(fam)
with pd.option_context("display.float_format", lambda v: f"{v:.3e}", "display.width", 160):
    display(fam)

lines = [r"\begin{tabular}{@{}llrrrrrr@{}}", r"\toprule",
         "Target & $L$ & " + " & ".join(lbl for _, lbl in FAMILIES) + r" & $\mathcal{O}$ opt. & Full opt. \\",
         r"\midrule"]
for t, tname in TARGETS:
    sub = fam[fam.target == t]
    for j, (_, r) in enumerate(sub.iterrows()):
        cells = [tname if j == 0 else "", str(int(r.L))]
        cells += [tex_sci(r[m]) for m, _ in FAMILIES if m != "spherical_odd"]
        # the null and the optima do not depend on depth: shown once per target
        cells += ([tex_sci(r.spherical_odd), tex_sci(r.ols_odd), tex_sci(r.ols_full)]
                  if j == 0 else ["", "", ""])
        lines.append(" & ".join(cells) + r" \\")
    if t != TARGETS[-1][0]:
        lines.append(r"\addlinespace[2pt]")
lines += [r"\bottomrule", r"\end{tabular}"]
(PAPER / "family_table.tex").write_text("% Generated by notebooks/paper2_analysis.ipynb -- do not edit by hand.\n"
                                        + "\n".join(lines) + "\n")
print(f"wrote {PAPER / 'family_table.tex'}")


## 7. Write `paper_values.tex` and check the manuscript

In [ ]:
vals = pd.DataFrame(VALUES)
dups = vals.macro[vals.macro.duplicated()].tolist()
assert not dups, f"duplicate macro names: {dups}"

lines = ["% Generated by notebooks/paper2_analysis.ipynb -- do not edit by hand.",
         f"% numpy {env['numpy']}, scipy {env['scipy']}, pandas {env['pandas']}",
         r"\providecommand{\textcolor}[2]{#2}", ""]
for r in vals.itertuples():
    lines.append(rf"\newcommand{{\pv{r.macro}}}{{{r.tex}}}  % {r.source}")
PAPER.mkdir(exist_ok=True)
(PAPER / "paper_values.tex").write_text("\n".join(lines) + "\n")
vals.to_csv(RESULTS / "paper_values.csv", index=False)
print(f"wrote {len(vals)} macros to paper/paper_values.tex ({len(MISSING)} missing)")

ms = PAPER / "main.tex"
if ms.exists():
    used = set(re.findall(r"\\pv([A-Za-z]+)", ms.read_text()))
    undefined = sorted(used - set(vals.macro))
    print("macros used in manuscript:", len(used))
    if undefined:
        print("UNDEFINED in manuscript:", undefined)
    if r"\input{paper_values}" not in ms.read_text():
        print("manuscript does not \\input{paper_values}")

In [ ]:
if MISSING:
    print("Values that could not be computed (shown as ?? in the PDF):")
    display(pd.DataFrame(MISSING))
else:
    print("All values computed.");